In [1]:
# ============================================================
# CHECKPOINT 0: Load 1 file nhỏ từ HuggingFace, verify schema
# Mục tiêu: biết đúng tên cột trước khi làm gì tiếp theo
# ============================================================

!pip install huggingface_hub pyarrow -q

from huggingface_hub import snapshot_download, list_repo_files
import pyarrow.parquet as pq
import pandas as pd
import os

REPO_ID   = "datdong2004/amazonNew-cleaned"
CATEGORY  = "tools_&_home_improvement"
HF_TOKEN  = None  # nếu repo private thì điền token vào đây

# ── Step 1: Xem repo có những file/folder gì ─────────────────
print("=== Cấu trúc repo ===")
try:
    files = list(list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN))
    for f in files[:30]:
        print(f)
    if len(files) > 30:
        print(f"... và {len(files)-30} files nữa")
except Exception as e:
    print(f"Lỗi: {e}")

=== Cấu trúc repo ===
.gitattributes
checkpoints.json
main_category=alexa_skills/._SUCCESS.crc
main_category=alexa_skills/_SUCCESS
main_category=alexa_skills/overall=1/.part-00000-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00001-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00002-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00003-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00004-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00005-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00006-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00007-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_

In [2]:
# ── Step 2: Download chỉ tools_&_home_improvement category ────────────────
# ignore_patterns để không kéo về categories khác — tiết kiệm thời gian và storage

LOCAL_DIR = f"/kaggle/working/data/{CATEGORY}"
os.makedirs(LOCAL_DIR, exist_ok=True)

snapshot_download(
    repo_id    = REPO_ID,
    repo_type  = "dataset",
    local_dir  = LOCAL_DIR,
    token      = HF_TOKEN,
    # Chỉ lấy Electronics, bỏ qua tất cả categories khác
    allow_patterns  = [f"*Tools_&_home_improvement*", f"*tools_&_home_improvement*"],
    ignore_patterns = ["*.json", "*.md", "*.txt"],
)

# Kiểm tra đã download được gì
downloaded = []
for root, dirs, fnames in os.walk(LOCAL_DIR):
    for f in fnames:
        if f.endswith(".parquet"):
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / 1024**2
            downloaded.append((full, size_mb))

print(f"\nĐã download: {len(downloaded)} files")
print(f"Tổng dung lượng: {sum(s for _, s in downloaded):.1f} MB")
for path, mb in downloaded[:5]:
    rel = os.path.relpath(path, LOCAL_DIR)
    print(f"  {rel:60s}  {mb:.1f} MB")

Fetching ... files: 0it [00:00, ?it/s]


Đã download: 40 files
Tổng dung lượng: 2000.7 MB
  main_category=tools_&_home_improvement/overall=4/part-00005-6b136033-5e10-4965-99f0-a7958bffc3fd.c000.zstd.parquet  40.2 MB
  main_category=tools_&_home_improvement/overall=4/part-00000-6b136033-5e10-4965-99f0-a7958bffc3fd.c000.zstd.parquet  39.9 MB
  main_category=tools_&_home_improvement/overall=4/part-00006-6b136033-5e10-4965-99f0-a7958bffc3fd.c000.zstd.parquet  40.2 MB
  main_category=tools_&_home_improvement/overall=4/part-00002-6b136033-5e10-4965-99f0-a7958bffc3fd.c000.zstd.parquet  40.3 MB
  main_category=tools_&_home_improvement/overall=4/part-00007-6b136033-5e10-4965-99f0-a7958bffc3fd.c000.zstd.parquet  40.4 MB


In [3]:
import pyarrow.dataset as ds
import pyarrow as pa
import pandas as pd

# Dùng ds.field trực tiếp, không qua pq.read_table
# explicit schema cho partition columns để fix type conflict

partition_schema = pa.schema([
    pa.field("overall", pa.int32()),
])

dataset = ds.dataset(
    LOCAL_DIR,
    format="parquet",
    partitioning=ds.partitioning(
        partition_schema,
        flavor="hive"  # tự parse overall=1/, overall=2/, ...
    ),
    # Fix main_category conflict: ép tất cả về string
    schema=None,
)

# Thử đọc 5 rows để confirm overall xuất hiện
df_test = dataset.head(5).to_pandas()
print("Columns:", df_test.columns.tolist())
print("\nOverall:", df_test["overall"].tolist() if "overall" in df_test.columns else "MISSING")
print("\nSample:")
print(df_test[["asin", "reviewerID", "overall", "review_date"]].to_string())

Columns: ['asin', 'reviewerID', 'reviewText', 'summary', 'title', 'price', 'brand', 'category', 'text', 'review_date', 'price_clean', 'flag_malformed_asin', 'flag_malformed_reviewer', 'flag_non_ascii_review', 'flag_malformed_overall', 'flag_malformed_date', 'main_category', 'flag_short_review', 'flag_no_prod_desc', 'flag_missing_price', 'flag_extreme_rating', 'overall']

Overall: [1, 1, 1, 1, 1]

Sample:
         asin      reviewerID  overall review_date
0  B00ZIRV39M  A3OLJMFZZB40FR        1  2017-05-23
1  B0000DCZ79  A2JNOY6S9GR1JK        1  2005-09-09
2  B012VLSYJE  A2OM2U6LE7BLNJ        1  2017-05-27
3  B00WZX2M38  A1O1ONFYOCMSFM        1  2017-07-10
4  B012I44L32  A2ZOBYN1A5EMQ6        1  2018-05-19


# **CHECKPOINT 1 — Load tools_&_home_improvement & Temporal Split**

In [4]:
import re

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"
NEEDED     = [COL_USER, COL_ITEM, COL_DATE]  # overall từ partition

# Load tất cả files, parse overall từ folder name
dfs = []
for path, size_mb in downloaded:
    match = re.search(r'overall=(\d)', path)
    overall_val = int(match.group(1)) if match else None
    pf = pq.ParquetFile(path)
    df_chunk = pf.read(columns=NEEDED).to_pandas()
    df_chunk[COL_RATING] = overall_val
    dfs.append(df_chunk)

df = pd.concat(dfs, ignore_index=True)

# Đảm bảo kiểu dữ liệu đúng
df[COL_DATE]   = pd.to_datetime(df[COL_DATE])
df[COL_RATING] = df[COL_RATING].astype(int)

print(f"Total reviews : {len(df):,}")
print(f"RAM           : ~{df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Null counts   :\n{df.isnull().sum()}")
print(f"\nRating distribution:")
print(df[COL_RATING].value_counts().sort_index())
print(f"\nDate range: {df[COL_DATE].min()} → {df[COL_DATE].max()}")

Total reviews : 5,524,598
RAM           : ~0.76 GB
Null counts   :
reviewerID     0
asin           0
review_date    0
overall        0
dtype: int64

Rating distribution:
overall
1     400498
2     245077
3     390082
4     833008
5    3655933
Name: count, dtype: int64

Date range: 1999-11-08 00:00:00 → 2018-10-04 00:00:00


In [5]:
# ============================================================
# K-CORE FILTERING — chạy trước CP1, thay thế df gốc
# ============================================================
K = 5

print(f"Trước filtering: {len(df):,} reviews, "
      f"{df[COL_USER].nunique():,} users, "
      f"{df[COL_ITEM].nunique():,} items")

df_core = df.copy()
iteration = 0

while True:
    iteration += 1
    
    # Đếm reviews per user và per item trong current df
    user_counts = df_core[COL_USER].value_counts()
    item_counts = df_core[COL_ITEM].value_counts()
    
    # Giữ lại chỉ users và items đủ K reviews
    valid_users = user_counts[user_counts >= K].index
    valid_items = item_counts[item_counts >= K].index
    
    df_filtered = df_core[
        df_core[COL_USER].isin(valid_users) &
        df_core[COL_ITEM].isin(valid_items)
    ]
    
    n_removed = len(df_core) - len(df_filtered)
    print(f"Iter {iteration}: removed {n_removed:,} reviews → {len(df_filtered):,} remaining")
    
    # Dừng khi không còn gì bị remove
    if n_removed == 0:
        break
    
    df_core = df_filtered

print(f"\nSau {K}-core filtering:")
print(f"  Reviews : {len(df_core):,}  ({len(df_core)/len(df):.1%} của original)")
print(f"  Users   : {df_core[COL_USER].nunique():,}")
print(f"  Items   : {df_core[COL_ITEM].nunique():,}")
print(f"\nReviews per user sau filtering:")
print(df_core.groupby(COL_USER).size().describe(percentiles=[.5,.75,.9,.95]))
print(f"\nReviews per item sau filtering:")
print(df_core.groupby(COL_ITEM).size().describe(percentiles=[.5,.75,.9,.95]))

Trước filtering: 5,524,598 reviews, 2,676,597 users, 142,570 items
Iter 1: removed 3,813,785 reviews → 1,710,813 remaining
Iter 2: removed 155,406 reviews → 1,555,407 remaining
Iter 3: removed 108,938 reviews → 1,446,469 remaining
Iter 4: removed 15,748 reviews → 1,430,721 remaining
Iter 5: removed 12,562 reviews → 1,418,159 remaining
Iter 6: removed 1,993 reviews → 1,416,166 remaining
Iter 7: removed 1,664 reviews → 1,414,502 remaining
Iter 8: removed 260 reviews → 1,414,242 remaining
Iter 9: removed 236 reviews → 1,414,006 remaining
Iter 10: removed 16 reviews → 1,413,990 remaining
Iter 11: removed 24 reviews → 1,413,966 remaining
Iter 12: removed 8 reviews → 1,413,958 remaining
Iter 13: removed 4 reviews → 1,413,954 remaining
Iter 14: removed 0 reviews → 1,413,954 remaining

Sau 5-core filtering:
  Reviews : 1,413,954  (25.6% của original)
  Users   : 170,424
  Items   : 58,535

Reviews per user sau filtering:
count    170424.000000
mean          8.296684
std           6.041496
min 

In [6]:
# Redo CP1 với df_core
df_core = df_core.sort_values(COL_DATE).reset_index(drop=True)

split_idx  = int(len(df_core) * 0.8)
split_date = df_core.iloc[split_idx][COL_DATE]

train = df_core.iloc[:split_idx].copy()
test  = df_core.iloc[split_idx:].copy()

train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test = test.copy()
test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)

mask_warm    = test["user_known"] & test["item_known"]
mask_cold_u  = ~test["user_known"] & test["item_known"]
mask_cold_i  = test["user_known"] & ~test["item_known"]
mask_cold_ui = ~test["user_known"] & ~test["item_known"]

n_users  = df_core[COL_USER].nunique()
n_items  = df_core[COL_ITEM].nunique()
sparsity = 1 - len(df_core) / (n_users * n_items)

print(f"Split date : {split_date}")
print(f"Train      : {len(train):,}")
print(f"Test       : {len(test):,}")
print(f"Sparsity   : {sparsity:.6%}")
print(f"\nWarm (both known)         : {mask_warm.sum():>8,}  ({mask_warm.mean():.1%})")
print(f"Cold user, warm item      : {mask_cold_u.sum():>8,}  ({mask_cold_u.mean():.1%})")
print(f"Warm user, cold item      : {mask_cold_i.sum():>8,}  ({mask_cold_i.mean():.1%})")
print(f"Fully cold (both unknown) : {mask_cold_ui.sum():>8,}  ({mask_cold_ui.mean():.1%})")

Split date : 2017-05-06 00:00:00
Train      : 1,131,163
Test       : 282,791
Sparsity   : 99.985826%

Warm (both known)         :  240,285  (85.0%)
Cold user, warm item      :   37,248  (13.2%)
Warm user, cold item      :    4,757  (1.7%)
Fully cold (both unknown) :      501  (0.2%)


After 5-core filtering and temporal split (2017-05-06), the test set contains 282,791 interactions, of which 85.0% are warm pairs (both user and item seen in training). The remaining 15.0% cold-start cases are dominated by cold-user, warm-item pairs (13.2%), indicating users whose activity appears only in the test window—an expected artifact given the low median user activity. Warm-user, cold-item pairs account for 1.7%, while fully cold pairs are negligible (0.2%). Cold-start scenarios are evaluated separately and addressed in Stage X.

# **CP2 — Compute Means**

In [7]:
global_mean = train[COL_RATING].mean()
global_std  = train[COL_RATING].std()
print(f"Global mean : {global_mean:.4f}")
print(f"Global std  : {global_std:.4f}")

user_stats = train.groupby(COL_USER)[COL_RATING].agg(["mean","count"])
user_stats.columns = ["user_mean_raw", "user_n"]

item_stats = train.groupby(COL_ITEM)[COL_RATING].agg(["mean","count"])
item_stats.columns = ["item_mean_raw", "item_n"]

print(f"\nUser review count — median: {user_stats['user_n'].median():.0f}, mean: {user_stats['user_n'].mean():.1f}")
print(f"Item review count — median: {item_stats['item_n'].median():.0f}, mean: {item_stats['item_n'].mean():.1f}")
print(f"\nUsers with 1 review: {(user_stats['user_n']==1).sum():,} ({(user_stats['user_n']==1).mean():.1%})")
print(f"Items with 1 review: {(item_stats['item_n']==1).sum():,} ({(item_stats['item_n']==1).mean():.1%})")

Global mean : 4.3812
Global std  : 1.0850

User review count — median: 6, mean: 6.9
Item review count — median: 9, mean: 19.5

Users with 1 review: 4,903 (3.0%)
Items with 1 review: 816 (1.4%)


In [8]:
# Bayesian shrinkage
m_user = float(user_stats["user_n"].median())
m_item = float(item_stats["item_n"].median())

user_stats["user_mean"] = (
    user_stats["user_n"] * user_stats["user_mean_raw"] + m_user * global_mean
) / (user_stats["user_n"] + m_user)

item_stats["item_mean"] = (
    item_stats["item_n"] * item_stats["item_mean_raw"] + m_item * global_mean
) / (item_stats["item_n"] + m_item)

print(f"m_user={m_user:.1f}, m_item={m_item:.1f}")
print(f"\nUser mean (Bayesian): {user_stats['user_mean'].describe()}")
print(f"\nItem mean (Bayesian): {item_stats['item_mean'].describe()}")

m_user=6.0, m_item=9.0

User mean (Bayesian): count    164581.000000
mean          4.376483
std           0.297875
min           2.095290
25%           4.219131
50%           4.449069
75%           4.607247
max           4.960075
Name: user_mean, dtype: float64

Item mean (Bayesian): count    58019.000000
mean         4.368107
std          0.253941
min          2.534461
25%          4.245032
50%          4.426528
75%          4.547682
max          4.953546
Name: item_mean, dtype: float64


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

user_mean_dict = user_stats["user_mean"].to_dict()
item_mean_dict = item_stats["item_mean"].to_dict()

def predict_vectorized(df_subset):
    u = df_subset[COL_USER].map(user_mean_dict).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_mean_dict).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0)

test["pred"]   = predict_vectorized(test)
test["actual"] = test[COL_RATING]

def evaluate(subset, label):
    if len(subset) == 0:
        print(f"{label:35s}: no samples")
        return
    rmse = mean_squared_error(subset["actual"], subset["pred"]) ** 0.5  # fix ở đây
    mae  = mean_absolute_error(subset["actual"], subset["pred"])
    print(f"{label:35s}  n={len(subset):>8,}  RMSE={rmse:.4f}  MAE={mae:.4f}")

print("="*75)
print("STAGE 0 — TOOLS_&_HOME_IMPROVEMENT BASELINE")
print("="*75)
evaluate(test,             "ALL TEST")
evaluate(test[mask_warm],  "Warm (both known)")
evaluate(test[mask_cold_u],"Cold user, warm item")
evaluate(test[mask_cold_i],"Warm user, cold item")
evaluate(test[mask_cold_ui],"Fully cold")

# Error distribution
errors = (test["actual"] - test["pred"]).abs()
print(f"\nError distribution:")
print(errors.describe(percentiles=[.5,.75,.9,.95,.99]))

big_err = (errors > 2.0).sum()
print(f"\nSai > 2 sao: {big_err:,} ({big_err/len(test):.2%})")

# Per-star breakdown (warm only)
print(f"\nRMSE per star (warm):")
warm = test[mask_warm]
for star in sorted(warm["actual"].unique()):
    s = warm[warm["actual"]==star]
    rmse = mean_squared_error(s["actual"], s["pred"]) ** 0.5
    mae  = mean_absolute_error(s["actual"], s["pred"])
    print(f"  ★{int(star)}  n={len(s):>7,}  RMSE={rmse:.4f}  MAE={mae:.4f}")

STAGE 0 — TOOLS_&_HOME_IMPROVEMENT BASELINE
ALL TEST                             n= 282,791  RMSE=1.0906  MAE=0.8058
Warm (both known)                    n= 240,285  RMSE=1.1029  MAE=0.8113
Cold user, warm item                 n=  37,248  RMSE=1.0142  MAE=0.7722
Warm user, cold item                 n=   4,757  RMSE=1.0569  MAE=0.7949
Fully cold                           n=     501  RMSE=0.9905  MAE=0.7775

Error distribution:
count    282791.000000
mean          0.805815
std           0.734958
min           0.000068
50%           0.553082
75%           0.743662
90%           1.571866
95%           2.950894
99%           3.471839
max           3.787625
dtype: float64

Sai > 2 sao: 26,082 (9.22%)

RMSE per star (warm):
  ★1  n= 14,055  RMSE=3.2607  MAE=3.2519
  ★2  n=  9,923  RMSE=2.2905  MAE=2.2807
  ★3  n= 16,456  RMSE=1.3291  MAE=1.3141
  ★4  n= 36,279  RMSE=0.4019  MAE=0.3692
  ★5  n=163,572  RMSE=0.5843  MAE=0.5599


Vấn đề: baseline predict gần như mọi thứ quanh 4.0–4.3 (vì global mean và hầu hết user/item means đều cluster ở đó). ★4 và ★5 chiếm ~79% warm test nên kéo overall RMSE xuống trông có vẻ ổn. Nhưng ★1 và ★2 bị predict sai hoàn toàn — đây là class imbalance problem.
Cold users (1.0986 RMSE) trông tốt hơn warm vì phần lớn cold users là người mới, có xu hướng rate cao (họ mua và hài lòng mới review), nên predict global_mean 4.23 cho họ lại ít sai hơn.

Kết luận cho Stage 0
Kết quả nằm trong expected range, không có bug. Đây là floor benchmark:
Warm RMSE : 1.2134  ← con số để beat ở stages sau
Warm MAE  : 0.9251
11.49% predictions sai hơn 2 sao — toàn bộ đến từ ★1 và ★2 bị predict thành ~4. Stages sau (MF, CF) phải cải thiện đáng kể trên low-rating predictions để được coi là thực sự tốt hơn baseline.

In [10]:
import json
from pathlib import Path

ARTIFACTS_DIR = Path("/kaggle/working/stage0_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

warm = test[mask_warm]

# Tính trước để không lặp lại lỗi
warm_rmse = mean_squared_error(warm["actual"], warm["pred"]) ** 0.5
warm_mae  = mean_absolute_error(warm["actual"], warm["pred"])
all_rmse  = mean_squared_error(test["actual"], test["pred"]) ** 0.5
all_mae   = mean_absolute_error(test["actual"], test["pred"])

# Split meta
split_meta = {
    "category"   : CATEGORY,
    "split_date" : str(split_date),
    "split_idx"  : int(split_idx),
    "n_train"    : len(train),
    "n_test"     : len(test),
    "global_mean": float(global_mean),
    "global_std" : float(global_std),
    "m_user"     : float(m_user),
    "m_item"     : float(m_item),
    "k_core"     : 5,
}
with open(ARTIFACTS_DIR / "split_meta.json", "w") as f:
    json.dump(split_meta, f, indent=2)

# Means
user_stats[["user_mean","user_mean_raw","user_n"]].to_parquet(
    ARTIFACTS_DIR / "user_means.parquet", compression="zstd"
)
item_stats[["item_mean","item_mean_raw","item_n"]].to_parquet(
    ARTIFACTS_DIR / "item_means.parquet", compression="zstd"
)

# Scores
scores = {
    "model"        : "simple_average_baseline",
    "category"     : CATEGORY,
    "warm_rmse"    : float(warm_rmse),
    "warm_mae"     : float(warm_mae),
    "all_rmse"     : float(all_rmse),
    "all_mae"      : float(all_mae),
    "n_warm_test"  : int(mask_warm.sum()),
    "n_total_test" : int(len(test)),
}
with open(ARTIFACTS_DIR / "baseline_scores.json", "w") as f:
    json.dump(scores, f, indent=2)

print("✓ split_meta.json")
print("✓ user_means.parquet")
print("✓ item_means.parquet")
print("✓ baseline_scores.json")
print(f"\nWarm RMSE : {warm_rmse:.4f}")
print(f"Warm MAE  : {warm_mae:.4f}")
print(f"All  RMSE : {all_rmse:.4f}")
print(f"All  MAE  : {all_mae:.4f}")
print(f"\nStage 0 complete — {CATEGORY.upper()}")

✓ split_meta.json
✓ user_means.parquet
✓ item_means.parquet
✓ baseline_scores.json

Warm RMSE : 1.1029
Warm MAE  : 0.8113
All  RMSE : 1.0906
All  MAE  : 0.8058

Stage 0 complete — TOOLS_&_HOME_IMPROVEMENT


Issue: The baseline predictions are heavily concentrated around ~4.0–4.3, driven by the global mean and the clustering of user/item means in that range. Since ★4 and ★5 dominate the warm test set (~83%), the overall RMSE appears reasonably low (1.1029 on warm). However, this masks a critical weakness: ★1 and ★2 are severely mispredicted, with extremely high errors (★1 RMSE = 3.26, ★2 RMSE = 2.29). This is a clear class imbalance problem, where the model fails to capture low-rating behavior.
Cold-start observation.
Cold-user cases (RMSE = 1.0136) perform better than warm users (1.1029), which is counterintuitive but expected. Most cold users are new and tend to leave positive reviews, so predicting near the global mean (~4.23) results in smaller errors. This creates an artificially optimistic impression of performance on cold users.

Conclusion for Stage 0:
The results fall within the expected range and indicate no major bugs. This serves as the baseline floor:
Warm RMSE: 1.1029 ← target to beat in later stages
Warm MAE: 0.8114
Additionally, 9.22% of predictions have errors greater than 2 stars, almost entirely due to ★1 and ★2 being predicted as ~4.
=> Future stages (MF, CF, etc.) must significantly improve performance on low-rating predictions; otherwise, gains in overall RMSE are not meaningful.

In [11]:
# ============================================
# SAVE TRAIN / TEST FOR LATER STAGES
# ============================================

# Chỉ lưu các cột cần cho Stage 2/3
train_to_save = train[[COL_USER, COL_ITEM, COL_RATING, COL_DATE]].copy()

test_to_save = test[[
    COL_USER, COL_ITEM, COL_RATING, COL_DATE,
    "user_known", "item_known"
]].copy()

test_to_save["is_warm"] = mask_warm.values
test_to_save["is_cold_user"] = mask_cold_u.values
test_to_save["is_cold_item"] = mask_cold_i.values
test_to_save["is_cold_both"] = mask_cold_ui.values

train_to_save.to_parquet(
    ARTIFACTS_DIR / "train.parquet",
    compression="zstd",
    index=False
)

test_to_save.to_parquet(
    ARTIFACTS_DIR / "test.parquet",
    compression="zstd",
    index=False
)

print("✓ train.parquet")
print("✓ test.parquet")

✓ train.parquet
✓ test.parquet


In [12]:
df_core[[COL_USER, COL_ITEM, COL_RATING, COL_DATE]].to_parquet(
    ARTIFACTS_DIR / "df_core.parquet",
    compression="zstd",
    index=False
)

print("✓ df_core.parquet")

✓ df_core.parquet


In [13]:
import pandas as pd
import json
from pathlib import Path

ARTIFACTS_DIR = Path("/kaggle/working/stage0_artifacts")

# Tính lại từ đầu
user_review_counts = (
    train.groupby(COL_USER)[COL_ITEM]
    .count()
    .reset_index(name="n_reviews")
)

bins   = [0, 1, 4, 19, float("inf")]
labels = ["new (0-1)", "cold (2-4)", "medium (5-19)", "warm (>=20)"]

user_review_counts["group"] = pd.cut(
    user_review_counts["n_reviews"],
    bins=bins,
    labels=labels
).astype(str)

# Distribution
vc_norm  = user_review_counts["group"].value_counts(normalize=True).reindex(labels)
vc_count = user_review_counts["group"].value_counts().reindex(labels)

print("=== Cold-start Profile — TOOLS_&_HOME_IMPROVEMENT Train Set ===")
print(f"Total users: {len(user_review_counts):,}\n")
for group in labels:
    ratio = vc_norm[group]
    count = vc_count[group]
    bar   = "█" * int(ratio * 40)
    print(f"{group:>15s}  {ratio:.1%}  ({count:>6,})  {bar}")

new_cold = vc_norm["new (0-1)"] + vc_norm["cold (2-4)"]
print(f"\nNew + Cold : {new_cold:.1%} → "
      f"{'Content/Popularity backbone quan trọng' if new_cold > 0.4 else 'MF viable làm chính'}")
print(f"Medium + Warm: {1-new_cold:.1%}")

# Save
profile_dict = {
    "category"          : CATEGORY,
    "total_train_users" : int(len(user_review_counts)),
    "distribution"      : {k: float(vc_norm[k]) for k in labels},
    "counts"            : {k: int(vc_count[k])  for k in labels},
}
with open(ARTIFACTS_DIR / "coldstart_profile.json", "w") as f:
    json.dump(profile_dict, f, indent=2)

user_review_counts.to_parquet(
    ARTIFACTS_DIR / "user_groups.parquet", compression="zstd"
)

print("\n✓ coldstart_profile.json")
print("✓ user_groups.parquet")
print(f"\nStage 1 complete — {CATEGORY.upper()}")

=== Cold-start Profile — TOOLS_&_HOME_IMPROVEMENT Train Set ===
Total users: 164,581

      new (0-1)  3.0%  ( 4,903)  █
     cold (2-4)  22.7%  (37,280)  █████████
  medium (5-19)  71.7%  (117,982)  ████████████████████████████
    warm (>=20)  2.7%  ( 4,416)  █

New + Cold : 25.6% → MF viable làm chính
Medium + Warm: 74.4%

✓ coldstart_profile.json
✓ user_groups.parquet

Stage 1 complete — TOOLS_&_HOME_IMPROVEMENT


# Stage 2

In [14]:
# ============================================================
# STAGE 2 — Cell 1: Reload data với text columns
# Chỉ load các cột cần thiết để tiết kiệm RAM
# ============================================================
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import re

NEEDED_S2 = ["asin", "reviewerID", "reviewText", "text", "title", "brand", "price_clean"]

dfs_s2 = []
for path, _ in downloaded:
    match = re.search(r'overall=(\d)', path)
    overall_val = int(match.group(1)) if match else None
    pf = pq.ParquetFile(path)
    df_chunk = pf.read(columns=NEEDED_S2).to_pandas()
    df_chunk[COL_RATING] = overall_val
    dfs_s2.append(df_chunk)

df_full = pd.concat(dfs_s2, ignore_index=True)

# Chỉ giữ lại rows có trong df_core (đã qua 5-core)
core_pairs = set(zip(df_core[COL_USER], df_core[COL_ITEM]))
mask = pd.Series(
    zip(df_full[COL_USER], df_full[COL_ITEM])
).isin(core_pairs)
df_full = df_full[mask.values].reset_index(drop=True)

print(f"df_full shape : {df_full.shape}")
print(f"RAM           : ~{df_full.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Null reviewText: {df_full['reviewText'].isna().sum():,}")
print(f"Null text      : {df_full['text'].isna().sum():,}")
print(f"Null title     : {df_full['title'].isna().sum():,}")
print(f"Sample text col: {df_full['text'].dropna().iloc[0][:100]}")

df_full shape : (1413954, 8)
RAM           : ~2.18 GB
Null reviewText: 0
Null text      : 0
Null title     : 0
Sample text col: filters and accessory items for 6000 series respirators description prefilter retainer


In [15]:
import gc

# Giữ lại đúng những gì cần, drop phần còn lại
# df_full cần giữ nhưng drop các cột không dùng trong Stage 2
KEEP_COLS = [COL_USER, COL_ITEM, COL_RATING,
             "reviewText", "text", "title", "brand", "price_clean"]

df_full = df_full[KEEP_COLS]

# df_core không cần nữa — mọi filtering đã xong
del df_core
gc.collect()

# Kiểm tra RAM sau khi dọn
import psutil
ram = psutil.virtual_memory()
print(f"RAM used  : {ram.used/1e9:.1f} GB")
print(f"RAM avail : {ram.available/1e9:.1f} GB")
print(f"RAM total : {ram.total/1e9:.1f} GB")
print(f"\ndf_full shape : {df_full.shape}")
print(f"df_full RAM   : ~{df_full.memory_usage(deep=True).sum()/1e9:.2f} GB")

# text column: confirm là product description
print(f"\n=== text column sample ===")
print(df_full["text"].dropna().iloc[0][:200])
print(f"\n=== title column sample ===")
print(df_full["title"].dropna().iloc[0][:100])
print(f"\n=== brand sample ===")
print(df_full["brand"].value_counts().head(10))

RAM used  : 6.2 GB
RAM avail : 27.0 GB
RAM total : 33.7 GB

df_full shape : (1413954, 8)
df_full RAM   : ~2.18 GB

=== text column sample ===
filters and accessory items for 6000 series respirators description prefilter retainer

=== title column sample ===
3M Filter Retainer 501 , Respiratory Protection System Component

=== brand sample ===
brand
DEWALT          37314
TEKTON          19498
Stanley         18298
Bosch           13090
Leviton         12038
DELTA FAUCET    11818
BLACK+DECKER    10792
Moen            10418
Makita          10411
Honeywell       10258
Name: count, dtype: int64


In [16]:
# ============================================================
# STAGE 2 — Cell 2: Item Profiles — TF-IDF
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import numpy as np
import gc

# Chỉ dùng train rows
train_ids = set(zip(train[COL_USER], train[COL_ITEM]))
mask_train = pd.Series(
    zip(df_full[COL_USER], df_full[COL_ITEM])
).isin(train_ids)
train_full = df_full[mask_train.values].copy()
print(f"Train rows với text: {len(train_full):,}")

# ── Phần 1: Aggregate reviewText per item ────────────────────
item_reviews = (
    train_full.groupby(COL_ITEM)["reviewText"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index(name="aggregated_review")
)
print(f"Items có reviews: {len(item_reviews):,}")

tfidf_review = TfidfVectorizer(
    max_features=3000,
    max_df=0.95,
    min_df=5,
    ngram_range=(1, 2),
    sublinear_tf=True
)
review_matrix = tfidf_review.fit_transform(item_reviews["aggregated_review"])
print(f"Review TF-IDF : {review_matrix.shape}, "
      f"RAM ~{review_matrix.data.nbytes/1e9:.2f} GB")

# ── Phần 2: TF-IDF từ title + text per item ──────────────────
item_meta = (
    train_full[["asin", "title", "text"]]
    .drop_duplicates("asin")
    .set_index("asin")
    .reindex(item_reviews["asin"])   # align với item_reviews order
    .reset_index()
)
item_meta["meta_text"] = (
    item_meta["title"].fillna("") + " " +
    item_meta["text"].fillna("")
).str.strip()

tfidf_meta = TfidfVectorizer(
    max_features=1000,
    min_df=3,
    sublinear_tf=True
)
meta_matrix = tfidf_meta.fit_transform(item_meta["meta_text"])
print(f"Meta TF-IDF   : {meta_matrix.shape}, "
      f"RAM ~{meta_matrix.data.nbytes/1e9:.2f} GB")

# ── Phần 3: Numerical + Brand features ───────────────────────
items = (
    train_full.groupby(COL_ITEM)
    .agg(
        avg_rating  = (COL_RATING, "mean"),
        n_reviews   = (COL_RATING, "count"),
        price_clean = ("price_clean", "first"),
        brand       = ("brand", "first"),
    )
    .reindex(item_reviews["asin"])   # align order
    .reset_index()
)

price_vals   = items["price_clean"].fillna(items["price_clean"].median())
alpha_price  = 1.0 / max(price_vals.mean(), 1e-6)
alpha_rating = 1.0 / max(items["avg_rating"].mean(), 1e-6)

numerical = csr_matrix(np.column_stack([
    price_vals.values * alpha_price,
    items["avg_rating"].fillna(3.5).values * alpha_rating,
    np.log1p(items["n_reviews"].values),
]))

# Top 100 brands → one-hot
top_brands = items["brand"].value_counts().head(100).index
items["brand_clean"] = items["brand"].where(
    items["brand"].isin(top_brands), other="other"
)
brand_dummies = csr_matrix(
    pd.get_dummies(items["brand_clean"], prefix="brand")
    .values.astype(float)
)
print(f"Numerical     : {numerical.shape}")
print(f"Brand dummies : {brand_dummies.shape}")

# ── Ghép toàn bộ ─────────────────────────────────────────────
item_profiles = hstack([
    review_matrix,
    meta_matrix,
    numerical,
    brand_dummies,
]).tocsr()

item_enc_s2 = {asin: idx for idx, asin
               in enumerate(item_reviews["asin"])}

print(f"\nItem profiles : {item_profiles.shape}")
print(f"nnz           : {item_profiles.nnz:,}")
print(f"RAM estimate  : ~{item_profiles.data.nbytes/1e9:.3f} GB")

# Dọn RAM
del train_full, item_meta
gc.collect()

ram = psutil.virtual_memory()
print(f"\nRAM available sau Cell 2: {ram.available/1e9:.1f} GB")

Train rows với text: 1,131,393
Items có reviews: 58,019
Review TF-IDF : (58019, 3000), RAM ~0.18 GB
Meta TF-IDF   : (58019, 1000), RAM ~0.02 GB
Numerical     : (58019, 3)
Brand dummies : (58019, 101)

Item profiles : (58019, 4104)
nnz           : 25,859,134
RAM estimate  : ~0.207 GB

RAM available sau Cell 2: 25.7 GB


In [17]:
# ============================================================
# STAGE 2 — Cell 3: Explicit Utility Matrix, double normalize
# ============================================================
import scipy.sparse as sp

user_list = train[COL_USER].unique()
item_list = train[COL_ITEM].unique()

user_enc_s2    = {u: i for i, u in enumerate(user_list)}
item_enc_s2_cf = {a: i for i, a in enumerate(item_list)}

rows = train[COL_USER].map(user_enc_s2).values
cols = train[COL_ITEM].map(item_enc_s2_cf).values
vals = train[COL_RATING].astype(float).values

M = sp.csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_list), len(item_list))
)
print(f"Utility matrix : {M.shape}, nnz={M.nnz:,}, fill={M.nnz/(M.shape[0]*M.shape[1]):.4%}")

# Bước 1: trừ user mean (vectorized)
M_csr       = M.copy().astype(float)
user_sums   = np.array(M_csr.sum(axis=1)).flatten()
user_counts = np.diff(M_csr.indptr).astype(float)
user_counts[user_counts == 0] = 1
user_means_vec = user_sums / user_counts

row_indices    = np.repeat(np.arange(M_csr.shape[0]), np.diff(M_csr.indptr))
M_csr.data    -= user_means_vec[row_indices]

# Bước 2: trừ item mean (vectorized)
M_csc       = M_csr.tocsc()
item_sums   = np.array(M_csc.sum(axis=0)).flatten()
item_counts = np.diff(M_csc.indptr).astype(float)
item_counts[item_counts == 0] = 1
item_means_vec = item_sums / item_counts

col_indices    = np.repeat(np.arange(M_csc.shape[1]), np.diff(M_csc.indptr))
M_csc.data    -= item_means_vec[col_indices]

M_norm = M_csc.tocsr()

print(f"M_norm shape   : {M_norm.shape}")
print(f"Mean of data   : {M_norm.data.mean():.8f}  (phải gần 0)")
print(f"Data range     : [{M_norm.data.min():.3f}, {M_norm.data.max():.3f}]")

ram = psutil.virtual_memory()
print(f"RAM available  : {ram.available/1e9:.1f} GB")

Utility matrix : (164581, 58019), nnz=1,128,092, fill=0.0118%
M_norm shape   : (164581, 58019)
Mean of data   : 0.00000000  (phải gần 0)
Data range     : [-7.219, 26.242]
RAM available  : 25.7 GB


In [18]:
# ============================================================
# STAGE 2 — Cell 4: Implicit Matrix
# ============================================================

C = sp.csr_matrix(
    (np.ones(len(rows)), (rows, cols)),
    shape=(len(user_list), len(item_list))
)
print(f"Implicit matrix : {C.shape}, nnz={C.nnz:,}")
assert C.nnz == M.nnz, "Mismatch encoding"
print("✓ Consistent với explicit matrix")

Implicit matrix : (164581, 58019), nnz=1,128,092
✓ Consistent với explicit matrix
